# NYPD Complaint Data — Complete Analysis & Prediction

**Dataset:** NYPD Complaint Data Current (Year To Date) — 2026-01-26  

## Pipeline
1. Load & Understand the dataset
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Prediction with **LightGBM** and **CatBoost** across **5 target variables**
6. Grand Comparison of all 10 model runs

## Why LightGBM + CatBoost?
| Model | Why it's better than RF / XGBoost |
|-------|-----------------------------------|
| **LightGBM** | Leaf-wise tree growth (vs level-wise) → faster convergence, better accuracy on large data. Histogram-based splitting handles 400k+ rows efficiently. |
| **CatBoost** | Native categorical encoding (ordered target statistics) eliminates label-encoding artifacts. Built-in overfitting detection. Usually best out-of-box accuracy on categorical-heavy tabular data like this. |

## 5 Target Variables
| # | Target | Task | Real-world value |
|---|--------|------|------------------|
| 1 | `LAW_CAT_CD` | 3-class (Felony / Misdemeanor / Violation) | Predict crime **severity** → triage police resources |
| 2 | `CRM_ATPT_CPTD_CD` | Binary (Completed / Attempted) | Predict crime **outcome** → prevention strategy |
| 3 | `BORO_NM` | 5-class (borough) | Predict **where** a crime occurs → geographic resource allocation |
| 4 | `OFNS_DESC` (top 10) | 10-class (offense type) | Predict **what kind** of crime → specialised unit deployment |
| 5 | `DAILY_CASE_COUNT` | Regression | Predict **how many** crimes per day → staffing & planning |

---
## 1. Setup & Imports

In [ ]:
# Install CatBoost if not already present (LightGBM is already installed)
!pip install catboost lightgbm --quiet

In [ ]:
# ── Core libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── ML libraries ──
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score
)

# ── Advanced gradient boosting models ──
import lightgbm as lgb       # Leaf-wise boosting, fast on large data
from catboost import CatBoostClassifier, CatBoostRegressor  # Native categorical handling

# ── Plotting config ──
sns.set_theme(style='whitegrid', font_scale=1.05)
pd.set_option('display.max_columns', 40)

print('All libraries loaded successfully.')

---
## 2. Load Dataset

In [ ]:
# ── Update this path if running on a different machine ──
DATASET_PATH = '/Users/rahulraj1406/DADM/NYPD_Complaint_Data_Current_(Year_To_Date)_20260126.csv'

df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
print(f'Loaded: {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns')

---
## 3. Understand the Dataset

In [ ]:
# ── Shape & column names ──
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')
print(f'\nColumn names:\n{df_raw.columns.tolist()}')

In [ ]:
# ── First 5 rows ──
df_raw.head()

In [ ]:
# ── Last 5 rows — helps spot truncation or data quality issues at the end ──
df_raw.tail()

In [ ]:
# ── Random sample — gives a feel for the variety in the data ──
df_raw.sample(5, random_state=42)

In [ ]:
# ── Data types & non-null counts ──
df_raw.info()

In [ ]:
# ── Numeric column statistics ──
df_raw.describe()

In [ ]:
# ── Missing values: count + percentage, sorted descending ──
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})
print(miss_df[miss_df['Count'] > 0].to_string())  # only show columns with missing data

In [ ]:
# ── Unique values per column — helps identify high-cardinality vs low-cardinality cols ──
df_raw.nunique().sort_values(ascending=False)

---
## 4. Data Cleaning

In [ ]:
# Work on a copy so we can always go back to raw if needed
df = df_raw.copy()
print(f'Starting shape: {df.shape}')

In [ ]:
# ── 4.1  Parse date & time columns ──
# Convert complaint start/end dates to proper datetime objects
df['CMPLNT_FR_DT'] = pd.to_datetime(df['CMPLNT_FR_DT'], errors='coerce')
df['CMPLNT_TO_DT'] = pd.to_datetime(df['CMPLNT_TO_DT'], errors='coerce')

# Parse complaint start time into a proper time object
df['CMPLNT_FR_TM'] = pd.to_datetime(df['CMPLNT_FR_TM'], format='%H:%M:%S', errors='coerce').dt.time

print('Dates and times parsed.')

In [ ]:
# ── 4.2  Remove duplicate rows ──
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed: {before - len(df)}')

In [ ]:
# ── 4.3  Remove invalid GPS coordinates ──
# NYC bounding box:  Lat 40.49 – 40.92,  Lon -74.26 – -73.70
# We use a slightly wider box (40 – 42, -75 – -72) to keep edge cases
df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

before = len(df)
df = df[df['Latitude'].between(40, 42) & df['Longitude'].between(-75, -72)]
print(f'Invalid coordinates removed: {before - len(df)}')

In [ ]:
# ── 4.4  Remove rows where complaint end date is before start date ──
mask = (df['CMPLNT_TO_DT'] - df['CMPLNT_FR_DT']).dt.days < 0
df = df[~mask]
print(f'Inconsistent date rows removed: {mask.sum()}')

In [ ]:
# ── 4.5  Drop columns that are >50% null or not useful for modelling ──
# TRANSIT_DISTRICT and HOUSING_PSA are >90% null
high_null = df.columns[df.isnull().mean() > 0.5].tolist()
print(f'High-null columns dropped (>50% missing): {high_null}')

# Columns with no predictive value: IDs, redundant geo text, report date
useless_cols = [
    'CMPLNT_NUM',       # unique complaint ID — not a feature
    'CMPLNT_TO_DT',     # 20k nulls and mostly same as start date
    'CMPLNT_TO_TM',     # same issue
    'Lat_Lon',          # text duplicate of Latitude/Longitude
    'New Georeferenced Column',  # same as above
    'RPT_DT',           # report date — nearly identical to complaint date
    'STATION_NAME',     # mostly blank for non-transit crimes
    'PARKS_NM',         # mostly blank for non-park crimes
    'HADEVELOPT',       # mostly "(null)" for non-housing crimes
    'PD_CD',            # numeric code — PD_DESC is the readable version
    'KY_CD',            # numeric code — OFNS_DESC is the readable version
    'X_COORD_CD', 'Y_COORD_CD',  # state plane coords — Lat/Lon is enough
]

drop_all = list(set(high_null + useless_cols))
df = df.drop(columns=[c for c in drop_all if c in df.columns])
print(f'Total columns dropped: {len(drop_all)}')

In [ ]:
# ── 4.6  Drop rows missing critical fields ──
before = len(df)
df = df.dropna(subset=['BORO_NM', 'OFNS_DESC', 'CMPLNT_FR_DT', 'LAW_CAT_CD'])
print(f'Rows dropped (missing key fields): {before - len(df)}')

In [ ]:
# ── 4.7  Standardise placeholder strings to NaN ──
# The dataset uses "(null)", "UNKNOWN", and "U" as missing-value markers in text columns
cat_cols = df.select_dtypes('object').columns
df[cat_cols] = df[cat_cols].replace({'(null)': np.nan, 'UNKNOWN': np.nan, 'U': np.nan})
print('Placeholder strings replaced with NaN.')

In [ ]:
# ── 4.8  Engineer time-based features from the complaint date/time ──
# These will be key predictors across all 5 targets

# HOUR of day (0-23) — crime patterns shift dramatically by hour
df['HOUR'] = pd.to_datetime(
    df['CMPLNT_FR_TM'].astype(str), format='%H:%M:%S', errors='coerce'
).dt.hour

# MONTH (1-12) — captures seasonal patterns
df['MONTH'] = df['CMPLNT_FR_DT'].dt.month

# DAY_OF_WEEK (0=Mon, 6=Sun) — weekday vs weekend effects
df['DAY_OF_WEEK'] = df['CMPLNT_FR_DT'].dt.dayofweek

# IS_WEEKEND — binary flag, simpler signal for models
df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 5).astype(int)

# TIME_BUCKET — group hours into intuitive periods
#   0 = Night (00-05), 1 = Morning (06-11), 2 = Afternoon (12-17), 3 = Evening (18-23)
df['TIME_BUCKET'] = pd.cut(
    df['HOUR'], bins=[-1, 5, 11, 17, 23],
    labels=[0, 1, 2, 3]  # Night, Morning, Afternoon, Evening
).astype(float).astype('Int64')

# YEAR — in case dataset spans multiple years
df['YEAR'] = df['CMPLNT_FR_DT'].dt.year

print(f'\nFinal clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

---
## 5. Exploratory Data Analysis (EDA)

### 5.1  Temporal Patterns — When do crimes happen?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Monthly complaints ──
monthly = df['MONTH'].value_counts().sort_index()
axes[0].bar(monthly.index, monthly.values, color='steelblue')
axes[0].set_title('Complaints by Month', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(1, 13))

# ── Day of week complaints ──
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = df['DAY_OF_WEEK'].value_counts().sort_index()
colors_dow = ['coral' if d >= 5 else '#6baed6' for d in dow.index]  # highlight weekends
axes[1].bar(day_labels, dow.values, color=colors_dow)
axes[1].set_title('Complaints by Day of Week', fontweight='bold')
axes[1].set_xlabel('Day (red = weekend)')

# ── Hourly complaints ──
hourly = df['HOUR'].value_counts().sort_index()
axes[2].fill_between(hourly.index, hourly.values, alpha=0.3, color='seagreen')
axes[2].plot(hourly.index, hourly.values, marker='o', color='seagreen', markersize=4)
axes[2].set_title('Complaints by Hour of Day', fontweight='bold')
axes[2].set_xlabel('Hour (0-23)')
axes[2].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

In [ ]:
# ── Daily complaint trend line — shows volume over time ──
daily_vol = df.groupby(df['CMPLNT_FR_DT'].dt.date).size()

plt.figure(figsize=(14, 4))
plt.plot(daily_vol.index, daily_vol.values, linewidth=0.6, color='steelblue')
# Add 7-day rolling average to smooth out noise
rolling_avg = daily_vol.rolling(7).mean()
plt.plot(rolling_avg.index, rolling_avg.values, color='red', linewidth=1.5, label='7-day avg')
plt.title('Daily Complaint Volume Over Time', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Complaints per Day')
plt.legend()
plt.tight_layout()
plt.show()

### 5.2  Categorical Distributions — What, Where, How Severe?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# ── Top 15 Offense Types ──
top15 = df['OFNS_DESC'].value_counts().dropna().nlargest(15)
sns.barplot(x=top15.values, y=top15.index, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Top 15 Offense Types', fontweight='bold')
axes[0, 0].set_xlabel('Count')

# ── Complaints by Borough ──
boro = df['BORO_NM'].value_counts().dropna()
sns.barplot(x=boro.index, y=boro.values, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Complaints by Borough', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=20)

# ── Crime Severity (our Target #1) ──
severity = df['LAW_CAT_CD'].value_counts().dropna()
colors_sev = ['#d62728', '#ff7f0e', '#2ca02c']  # red=felony, orange=misd, green=viol
axes[1, 0].pie(severity.values, labels=severity.index,
               autopct='%1.1f%%', startangle=140, colors=colors_sev[:len(severity)])
axes[1, 0].set_title('Crime Severity (Target #1)', fontweight='bold')

# ── Completed vs Attempted (our Target #2) ──
outcome = df['CRM_ATPT_CPTD_CD'].value_counts().dropna()
axes[1, 1].bar(outcome.index, outcome.values, color=['#1f77b4', '#ff7f0e'])
axes[1, 1].set_title('Completed vs Attempted (Target #2)', fontweight='bold')
for i, (lbl, val) in enumerate(zip(outcome.index, outcome.values)):
    axes[1, 1].text(i, val + 1000, f'{val:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.3  Bivariate Analysis — Relationships between variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Crime severity by Borough — shows which boroughs have more felonies ──
ct = pd.crosstab(df['BORO_NM'], df['LAW_CAT_CD']).dropna()
ct.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')
axes[0].set_title('Crime Severity by Borough', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Severity')

# ── Completed vs Attempted by Borough ──
ct2 = pd.crosstab(df['BORO_NM'], df['CRM_ATPT_CPTD_CD']).dropna()
ct2.plot(kind='bar', stacked=False, ax=axes[1], colormap='Set1')
axes[1].set_title('Completed vs Attempted by Borough', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Outcome')

plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: Top 10 Offense Types × Hour of Day ──
# This reveals when specific crime types peak during the day
top10_ofns = df['OFNS_DESC'].value_counts().dropna().nlargest(10).index
df_heat = df[df['OFNS_DESC'].isin(top10_ofns)]
pt = pd.crosstab(df_heat['OFNS_DESC'], df_heat['HOUR'])

plt.figure(figsize=(16, 7))
sns.heatmap(pt, cmap='YlOrRd', linewidths=0.3, cbar_kws={'label': 'Complaint Count'})
plt.title('Top 10 Offense Types × Hour of Day', fontweight='bold')
plt.xlabel('Hour')
plt.ylabel('Offense Type')
plt.tight_layout()
plt.show()

In [ ]:
# ── Geographic scatter: 20k sample coloured by severity ──
sample_geo = df[['Latitude', 'Longitude', 'LAW_CAT_CD']].dropna().sample(
    min(20000, len(df)), random_state=42
)

plt.figure(figsize=(10, 9))
color_map = {'FELONY': '#d62728', 'MISDEMEANOR': '#ff7f0e', 'VIOLATION': '#2ca02c'}
for cat in ['VIOLATION', 'MISDEMEANOR', 'FELONY']:  # plot felonies last so they're on top
    grp = sample_geo[sample_geo['LAW_CAT_CD'] == cat]
    if len(grp) > 0:
        plt.scatter(grp['Longitude'], grp['Latitude'], s=0.4, alpha=0.35,
                    label=cat, color=color_map.get(cat, 'gray'))
plt.title('NYC Crime Locations (20k sample) by Severity', fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(markerscale=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 Premise Types ──
plt.figure(figsize=(10, 6))
top10_prem = df['PREM_TYP_DESC'].value_counts().dropna().nlargest(10)
sns.barplot(x=top10_prem.values, y=top10_prem.index, palette='Oranges_r')
plt.title('Top 10 Premise Types', fontweight='bold')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

---
## 6. Feature Engineering & ML Helper Functions

We define reusable helper functions here so that Sections 7–11 are clean and consistent.

In [ ]:
# ── Feature columns for classification tasks (Targets 1-4) ──
# These are the features that will be available to predict all classification targets.
# The target column itself is excluded automatically inside prepare_data().

CLASSIFICATION_FEATURES = [
    'BORO_NM',             # Borough (5 categories)
    'ADDR_PCT_CD',         # Precinct code (numeric, ~77 unique)
    'OFNS_DESC',           # Offense description (high cardinality)
    'PD_DESC',             # Specific offense sub-description
    'PREM_TYP_DESC',      # Premise type (street, residence, etc.)
    'LOC_OF_OCCUR_DESC',  # Inside/outside/etc.
    'PATROL_BORO',        # Patrol borough (administrative grouping)
    'JURISDICTION_CODE',  # Which agency has jurisdiction
    'SUSP_AGE_GROUP',     # Suspect age group
    'SUSP_RACE',          # Suspect race
    'SUSP_SEX',           # Suspect sex
    'VIC_AGE_GROUP',      # Victim age group
    'VIC_RACE',           # Victim race
    'VIC_SEX',            # Victim sex
    'HOUR',               # Hour of day (0-23)
    'MONTH',              # Month (1-12)
    'DAY_OF_WEEK',        # Day of week (0=Mon, 6=Sun)
    'IS_WEEKEND',         # Weekend flag
    'TIME_BUCKET',        # Night/Morning/Afternoon/Evening
    'Latitude',           # GPS latitude
    'Longitude',          # GPS longitude
]

# Only keep columns that actually exist in the cleaned dataframe
CLASSIFICATION_FEATURES = [f for f in CLASSIFICATION_FEATURES if f in df.columns]
print(f'{len(CLASSIFICATION_FEATURES)} feature columns available for classification.')

In [ ]:
def prepare_classification_data(target_col, features, dataframe, top_n_target=None):
    """
    Prepare X, y for classification:
      - Drops rows with NaN in features or target
      - Optionally filters to top_n most frequent target classes
      - Removes the target itself from features if present
      - Label-encodes all object columns
    
    Returns: X, y, label_encoders_dict, feature_names_used
    """
    # Remove target from features if it accidentally got included
    feats = [f for f in features if f != target_col]
    cols = feats + [target_col]
    data = dataframe[cols].dropna().copy()
    
    # Optionally keep only the top N classes (for high-cardinality targets like OFNS_DESC)
    if top_n_target is not None:
        top_classes = data[target_col].value_counts().nlargest(top_n_target).index
        data = data[data[target_col].isin(top_classes)]
        print(f'  Filtered to top {top_n_target} classes: {list(top_classes)}')
    
    # Label-encode all object/string columns (including the target)
    le_dict = {}
    for col in data.select_dtypes('object').columns:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))
        le_dict[col] = le
    
    X = data[feats].values
    y = data[target_col].values
    print(f'  Samples: {len(data):,}  |  Classes: {len(np.unique(y))}  |  Features: {len(feats)}')
    return X, y, le_dict, feats


def train_and_evaluate(X_train, y_train, X_test, y_test,
                       model, model_name, class_names=None):
    """
    Train model, predict, print classification report, return results dict.
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f'\n{"=" * 50}')
    print(f'  {model_name}')
    print(f'  Accuracy : {acc:.4f}  |  F1: {f1:.4f}  |  Precision: {prec:.4f}  |  Recall: {rec:.4f}')
    print(f'{"=" * 50}')
    print(classification_report(y_test, y_pred, target_names=class_names))
    
    return {
        'model_name': model_name,
        'accuracy': acc, 'f1_weighted': f1,
        'precision': prec, 'recall': rec,
        'y_pred': y_pred, 'model_obj': model
    }


def plot_comparison(res_lgb, res_cat, y_test, class_names, target_label):
    """
    3-panel plot: metrics bar chart + 2 confusion matrices.
    """
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    
    # ── Metrics bar chart ──
    metrics = pd.DataFrame([
        {'Model': 'LightGBM', 'Accuracy': res_lgb['accuracy'], 'F1': res_lgb['f1_weighted']},
        {'Model': 'CatBoost', 'Accuracy': res_cat['accuracy'], 'F1': res_cat['f1_weighted']},
    ]).set_index('Model')
    metrics.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
    axes[0].set_title(f'{target_label}: Accuracy & F1', fontweight='bold')
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis='x', rotation=0)
    # Add value labels on bars
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.3f', fontsize=8)
    
    # ── Confusion matrices ──
    for i, (res, name) in enumerate([(res_lgb, 'LightGBM'), (res_cat, 'CatBoost')]):
        cm = confusion_matrix(y_test, res['y_pred'])
        disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
        disp.plot(ax=axes[i + 1], colorbar=False, cmap='Blues')
        axes[i + 1].set_title(f'{name} Confusion Matrix', fontweight='bold')
    
    plt.suptitle(f'Target: {target_label}', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def plot_feature_importance(model, feature_names, title):
    """
    Horizontal bar chart of feature importances.
    Works with LightGBM and CatBoost models.
    """
    imp = model.feature_importances_
    feat_imp = pd.Series(imp, index=feature_names).sort_values(ascending=True)
    
    plt.figure(figsize=(8, max(4, len(feature_names) * 0.3)))
    feat_imp.plot(kind='barh', color='teal', edgecolor='black', linewidth=0.3)
    plt.title(title, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

print('Helper functions ready.')

---
## 7. Target 1 — Crime Severity (`LAW_CAT_CD`)

**Task:** 3-class classification — FELONY / MISDEMEANOR / VIOLATION  
**Why:** Knowing the severity of an incoming crime lets dispatchers prioritise the right response.

In [ ]:
# ── Prepare data ──
print('Target 1: LAW_CAT_CD (Crime Severity)')
X1, y1, le1, feats1 = prepare_classification_data('LAW_CAT_CD', CLASSIFICATION_FEATURES, df)

# Stratified split preserves class proportions
X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)
class_names1 = le1['LAW_CAT_CD'].classes_ if 'LAW_CAT_CD' in le1 else None
print(f'Train: {X1_tr.shape}  |  Test: {X1_te.shape}')

In [ ]:
# ── LightGBM ──
# Leaf-wise growth + histogram binning = fast & accurate on 400k rows
lgb1 = lgb.LGBMClassifier(
    n_estimators=500,       # more trees for better convergence
    max_depth=8,            # moderate depth prevents overfitting
    learning_rate=0.05,     # smaller LR + more trees = better generalisation
    num_leaves=63,          # leaf-wise: more leaves → more expressive
    min_child_samples=50,   # regularisation: need 50+ samples per leaf
    subsample=0.8,          # row subsampling for variance reduction
    colsample_bytree=0.8,   # feature subsampling
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
res1_lgb = train_and_evaluate(X1_tr, y1_tr, X1_te, y1_te, lgb1, 'LightGBM — LAW_CAT_CD', class_names1)

In [ ]:
# ── CatBoost ──
# Handles categoricals natively via ordered target statistics
cat1 = CatBoostClassifier(
    iterations=500,         # number of boosting rounds
    depth=8,                # tree depth
    learning_rate=0.05,
    l2_leaf_reg=5,          # L2 regularisation on leaf values
    random_seed=42,
    verbose=0               # suppress training logs
)
res1_cat = train_and_evaluate(X1_tr, y1_tr, X1_te, y1_te, cat1, 'CatBoost — LAW_CAT_CD', class_names1)

In [ ]:
# ── Visual comparison ──
plot_comparison(res1_lgb, res1_cat, y1_te, class_names1, 'LAW_CAT_CD (Crime Severity)')
plot_feature_importance(res1_lgb['model_obj'], feats1, 'LightGBM Feature Importance — LAW_CAT_CD')

---
## 8. Target 2 — Crime Outcome (`CRM_ATPT_CPTD_CD`)

**Task:** Binary classification — COMPLETED vs ATTEMPTED  
**Why:** Predicting whether a crime will be completed helps identify which scenarios can be interrupted.

In [ ]:
print('Target 2: CRM_ATPT_CPTD_CD (Completed vs Attempted)')
X2, y2, le2, feats2 = prepare_classification_data('CRM_ATPT_CPTD_CD', CLASSIFICATION_FEATURES, df)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)
class_names2 = le2['CRM_ATPT_CPTD_CD'].classes_ if 'CRM_ATPT_CPTD_CD' in le2 else None
print(f'Train: {X2_tr.shape}  |  Test: {X2_te.shape}')

In [ ]:
# ── LightGBM ──
lgb2 = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
res2_lgb = train_and_evaluate(X2_tr, y2_tr, X2_te, y2_te, lgb2, 'LightGBM — CRM_ATPT_CPTD_CD', class_names2)

In [ ]:
# ── CatBoost ──
cat2 = CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    l2_leaf_reg=5, random_seed=42, verbose=0
)
res2_cat = train_and_evaluate(X2_tr, y2_tr, X2_te, y2_te, cat2, 'CatBoost — CRM_ATPT_CPTD_CD', class_names2)

In [ ]:
plot_comparison(res2_lgb, res2_cat, y2_te, class_names2, 'CRM_ATPT_CPTD_CD (Completed vs Attempted)')
plot_feature_importance(res2_lgb['model_obj'], feats2, 'LightGBM Feature Importance — CRM_ATPT_CPTD_CD')

---
## 9. Target 3 — Borough Prediction (`BORO_NM`)

**Task:** 5-class classification — MANHATTAN / BROOKLYN / BRONX / QUEENS / STATEN ISLAND  
**Why:** Given a crime's characteristics (type, time, severity, demographics), can we predict WHERE it happens? This reveals geographic crime patterns.

In [ ]:
# For BORO_NM prediction, we EXCLUDE Latitude/Longitude (they directly encode borough)
# and PATROL_BORO (it's basically the same info as borough)
boro_features = [f for f in CLASSIFICATION_FEATURES
                 if f not in ('BORO_NM', 'Latitude', 'Longitude', 'PATROL_BORO', 'ADDR_PCT_CD')]

print('Target 3: BORO_NM (Borough)')
print(f'Features (geo removed): {boro_features}')
X3, y3, le3, feats3 = prepare_classification_data('BORO_NM', boro_features, df)

X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)
class_names3 = le3['BORO_NM'].classes_ if 'BORO_NM' in le3 else None
print(f'Train: {X3_tr.shape}  |  Test: {X3_te.shape}')

In [ ]:
# ── LightGBM ──
lgb3 = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
res3_lgb = train_and_evaluate(X3_tr, y3_tr, X3_te, y3_te, lgb3, 'LightGBM — BORO_NM', class_names3)

In [ ]:
# ── CatBoost ──
cat3 = CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    l2_leaf_reg=5, random_seed=42, verbose=0
)
res3_cat = train_and_evaluate(X3_tr, y3_tr, X3_te, y3_te, cat3, 'CatBoost — BORO_NM', class_names3)

In [ ]:
plot_comparison(res3_lgb, res3_cat, y3_te, class_names3, 'BORO_NM (Borough)')
plot_feature_importance(res3_lgb['model_obj'], feats3, 'LightGBM Feature Importance — BORO_NM')

---
## 10. Target 4 — Offense Type (`OFNS_DESC`, top 10)

**Task:** 10-class classification — predict the type of crime committed  
**Why:** Knowing the likely offense type from situational data (location, time, demographics) helps dispatch the right specialised unit.

In [ ]:
# For OFNS_DESC prediction, we EXCLUDE PD_DESC (it's a sub-description of the offense)
# and KY_CD (numeric key for the same thing) — these would leak the answer
ofns_features = [f for f in CLASSIFICATION_FEATURES
                 if f not in ('OFNS_DESC', 'PD_DESC')]

print('Target 4: OFNS_DESC (Offense Type — top 10 classes)')
X4, y4, le4, feats4 = prepare_classification_data(
    'OFNS_DESC', ofns_features, df, top_n_target=10  # keep top 10 offense types only
)

X4_tr, X4_te, y4_tr, y4_te = train_test_split(
    X4, y4, test_size=0.2, random_state=42, stratify=y4
)
class_names4 = le4['OFNS_DESC'].classes_ if 'OFNS_DESC' in le4 else None
print(f'Train: {X4_tr.shape}  |  Test: {X4_te.shape}')

In [ ]:
# ── LightGBM ──
lgb4 = lgb.LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
res4_lgb = train_and_evaluate(X4_tr, y4_tr, X4_te, y4_te, lgb4, 'LightGBM — OFNS_DESC', class_names4)

In [ ]:
# ── CatBoost ──
cat4 = CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    l2_leaf_reg=5, random_seed=42, verbose=0
)
res4_cat = train_and_evaluate(X4_tr, y4_tr, X4_te, y4_te, cat4, 'CatBoost — OFNS_DESC', class_names4)

In [ ]:
plot_comparison(res4_lgb, res4_cat, y4_te, class_names4, 'OFNS_DESC (Offense Type — Top 10)')
plot_feature_importance(res4_lgb['model_obj'], feats4, 'LightGBM Feature Importance — OFNS_DESC')

---
## 11. Target 5 — Daily Case Count (Regression)

**Task:** Predict the total number of NYPD complaints filed on a given day  
**Why:** Forecasting daily crime volume helps with shift scheduling, overtime budgeting, and patrol planning.  
**Method:** Aggregate the row-level dataset into daily counts, then use temporal features to predict the count.

In [ ]:
# ── 11.1  Build the daily aggregated dataset ──
# Each row = one calendar day, target = total number of complaints that day

df_daily = df.assign(DATE=df['CMPLNT_FR_DT'].dt.date).dropna(subset=['DATE'])

# Count complaints per day
daily_counts = df_daily.groupby('DATE').size().reset_index(name='DAILY_CASE_COUNT')
daily_counts['DATE'] = pd.to_datetime(daily_counts['DATE'])

# ── Temporal features for each day ──
daily_counts['MONTH']       = daily_counts['DATE'].dt.month
daily_counts['DAY_OF_WEEK'] = daily_counts['DATE'].dt.dayofweek
daily_counts['IS_WEEKEND']  = (daily_counts['DAY_OF_WEEK'] >= 5).astype(int)
daily_counts['DAY_OF_MONTH']= daily_counts['DATE'].dt.day
daily_counts['WEEK_OF_YEAR']= daily_counts['DATE'].dt.isocalendar().week.astype(int)

# ── Lagged features: how many complaints happened 1, 7, and 14 days ago ──
# These capture short-term trends and weekly seasonality
daily_counts = daily_counts.sort_values('DATE')
daily_counts['LAG_1']  = daily_counts['DAILY_CASE_COUNT'].shift(1)   # yesterday's count
daily_counts['LAG_7']  = daily_counts['DAILY_CASE_COUNT'].shift(7)   # same day last week
daily_counts['LAG_14'] = daily_counts['DAILY_CASE_COUNT'].shift(14)  # two weeks ago

# ── Rolling averages: smoothed recent trends ──
daily_counts['ROLLING_7']  = daily_counts['DAILY_CASE_COUNT'].rolling(7).mean()   # weekly avg
daily_counts['ROLLING_30'] = daily_counts['DAILY_CASE_COUNT'].rolling(30).mean()  # monthly avg

# ── Borough-level daily breakdowns (how many per borough that day) ──
boro_daily = df_daily.groupby(['DATE', 'BORO_NM']).size().unstack(fill_value=0)
boro_daily.columns = [f'BORO_{c.replace(" ", "_")}' for c in boro_daily.columns]
boro_daily = boro_daily.reset_index()
boro_daily['DATE'] = pd.to_datetime(boro_daily['DATE'])
daily_counts = daily_counts.merge(boro_daily, on='DATE', how='left')

# ── Severity-level daily breakdowns ──
sev_daily = df_daily.groupby(['DATE', 'LAW_CAT_CD']).size().unstack(fill_value=0)
sev_daily.columns = [f'SEV_{c}' for c in sev_daily.columns]
sev_daily = sev_daily.reset_index()
sev_daily['DATE'] = pd.to_datetime(sev_daily['DATE'])
daily_counts = daily_counts.merge(sev_daily, on='DATE', how='left')

# Drop rows where lag features are NaN (first 14 days)
daily_counts = daily_counts.dropna()

print(f'Daily dataset: {daily_counts.shape}')
daily_counts.head()

In [ ]:
# ── 11.2  Prepare regression data ──
# Drop DATE and the per-borough/severity columns that leak the target
# (they sum up to DAILY_CASE_COUNT — that's data leakage)
leak_cols = [c for c in daily_counts.columns if c.startswith('BORO_') or c.startswith('SEV_')]

REG_FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'DAY_OF_MONTH', 'WEEK_OF_YEAR',
    'LAG_1', 'LAG_7', 'LAG_14', 'ROLLING_7', 'ROLLING_30'
]

X5 = daily_counts[REG_FEATURES].values
y5 = daily_counts['DAILY_CASE_COUNT'].values

# Time-series aware split: use last 20% chronologically (not random)
split_idx = int(len(X5) * 0.8)
X5_tr, X5_te = X5[:split_idx], X5[split_idx:]
y5_tr, y5_te = y5[:split_idx], y5[split_idx:]
dates_test = daily_counts['DATE'].values[split_idx:]

print(f'Regression features: {REG_FEATURES}')
print(f'Train: {X5_tr.shape}  |  Test: {X5_te.shape}')
print(f'Target range: {y5.min():.0f} – {y5.max():.0f} complaints/day')

In [ ]:
# ── LightGBM Regressor ──
lgb5 = lgb.LGBMRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb5.fit(X5_tr, y5_tr)
y5_pred_lgb = lgb5.predict(X5_te)

mae_lgb  = mean_absolute_error(y5_te, y5_pred_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y5_te, y5_pred_lgb))
r2_lgb   = r2_score(y5_te, y5_pred_lgb)

print(f'LightGBM Regression — Daily Case Count')
print(f'  MAE  : {mae_lgb:.2f} complaints/day')
print(f'  RMSE : {rmse_lgb:.2f}')
print(f'  R²   : {r2_lgb:.4f}')

In [ ]:
# ── CatBoost Regressor ──
cat5 = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=0
)
cat5.fit(X5_tr, y5_tr)
y5_pred_cat = cat5.predict(X5_te)

mae_cat  = mean_absolute_error(y5_te, y5_pred_cat)
rmse_cat = np.sqrt(mean_squared_error(y5_te, y5_pred_cat))
r2_cat   = r2_score(y5_te, y5_pred_cat)

print(f'CatBoost Regression — Daily Case Count')
print(f'  MAE  : {mae_cat:.2f} complaints/day')
print(f'  RMSE : {rmse_cat:.2f}')
print(f'  R²   : {r2_cat:.4f}')

In [ ]:
# ── 11.3  Regression visualisation ──

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Actual vs Predicted (LightGBM) — time series overlay ──
axes[0, 0].plot(dates_test, y5_te, label='Actual', color='black', linewidth=1)
axes[0, 0].plot(dates_test, y5_pred_lgb, label='LightGBM Predicted', color='steelblue', linewidth=1, alpha=0.8)
axes[0, 0].set_title('LightGBM: Actual vs Predicted Daily Cases', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Complaints / Day')

# ── Actual vs Predicted (CatBoost) — time series overlay ──
axes[0, 1].plot(dates_test, y5_te, label='Actual', color='black', linewidth=1)
axes[0, 1].plot(dates_test, y5_pred_cat, label='CatBoost Predicted', color='coral', linewidth=1, alpha=0.8)
axes[0, 1].set_title('CatBoost: Actual vs Predicted Daily Cases', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].set_ylabel('Complaints / Day')

# ── Scatter: Predicted vs Actual (both models) ──
axes[1, 0].scatter(y5_te, y5_pred_lgb, s=15, alpha=0.5, label='LightGBM', color='steelblue')
axes[1, 0].scatter(y5_te, y5_pred_cat, s=15, alpha=0.5, label='CatBoost', color='coral')
# Perfect prediction line
mn, mx = min(y5_te.min(), 500), max(y5_te.max(), 1800)
axes[1, 0].plot([mn, mx], [mn, mx], 'k--', linewidth=1, label='Perfect')
axes[1, 0].set_title('Predicted vs Actual', fontweight='bold')
axes[1, 0].set_xlabel('Actual')
axes[1, 0].set_ylabel('Predicted')
axes[1, 0].legend()

# ── Regression metrics comparison ──
metrics_reg = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'LightGBM': [mae_lgb, rmse_lgb, r2_lgb],
    'CatBoost': [mae_cat, rmse_cat, r2_cat]
}).set_index('Metric')
metrics_reg.plot(kind='bar', ax=axes[1, 1], color=['steelblue', 'coral'], edgecolor='black')
axes[1, 1].set_title('Regression Metrics Comparison', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=0)
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.2f', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance for regression ──
plot_feature_importance(lgb5, REG_FEATURES, 'LightGBM Feature Importance — Daily Case Count')

---
## 12. Grand Comparison — All 5 Targets × 2 Models

This section brings everything together: **10 model runs** side by side.

In [ ]:
# ── Build the master results table ──

grand_results = pd.DataFrame([
    # Target 1: Crime Severity
    {'Target': 'LAW_CAT_CD (Severity)',     'Model': 'LightGBM', 'Task': 'Classification',
     'Accuracy': res1_lgb['accuracy'], 'F1': res1_lgb['f1_weighted'],
     'Precision': res1_lgb['precision'], 'Recall': res1_lgb['recall']},
    {'Target': 'LAW_CAT_CD (Severity)',     'Model': 'CatBoost', 'Task': 'Classification',
     'Accuracy': res1_cat['accuracy'], 'F1': res1_cat['f1_weighted'],
     'Precision': res1_cat['precision'], 'Recall': res1_cat['recall']},
    
    # Target 2: Crime Outcome
    {'Target': 'CRM_ATPT_CPTD_CD (Outcome)', 'Model': 'LightGBM', 'Task': 'Classification',
     'Accuracy': res2_lgb['accuracy'], 'F1': res2_lgb['f1_weighted'],
     'Precision': res2_lgb['precision'], 'Recall': res2_lgb['recall']},
    {'Target': 'CRM_ATPT_CPTD_CD (Outcome)', 'Model': 'CatBoost', 'Task': 'Classification',
     'Accuracy': res2_cat['accuracy'], 'F1': res2_cat['f1_weighted'],
     'Precision': res2_cat['precision'], 'Recall': res2_cat['recall']},
    
    # Target 3: Borough
    {'Target': 'BORO_NM (Borough)',          'Model': 'LightGBM', 'Task': 'Classification',
     'Accuracy': res3_lgb['accuracy'], 'F1': res3_lgb['f1_weighted'],
     'Precision': res3_lgb['precision'], 'Recall': res3_lgb['recall']},
    {'Target': 'BORO_NM (Borough)',          'Model': 'CatBoost', 'Task': 'Classification',
     'Accuracy': res3_cat['accuracy'], 'F1': res3_cat['f1_weighted'],
     'Precision': res3_cat['precision'], 'Recall': res3_cat['recall']},
    
    # Target 4: Offense Type
    {'Target': 'OFNS_DESC (Offense Type)',   'Model': 'LightGBM', 'Task': 'Classification',
     'Accuracy': res4_lgb['accuracy'], 'F1': res4_lgb['f1_weighted'],
     'Precision': res4_lgb['precision'], 'Recall': res4_lgb['recall']},
    {'Target': 'OFNS_DESC (Offense Type)',   'Model': 'CatBoost', 'Task': 'Classification',
     'Accuracy': res4_cat['accuracy'], 'F1': res4_cat['f1_weighted'],
     'Precision': res4_cat['precision'], 'Recall': res4_cat['recall']},
    
    # Target 5: Daily Case Count (regression — use R² as "accuracy", MAE as supplementary)
    {'Target': 'Daily Case Count',           'Model': 'LightGBM', 'Task': 'Regression',
     'Accuracy': r2_lgb, 'F1': np.nan, 'Precision': np.nan, 'Recall': np.nan,
     'MAE': mae_lgb, 'RMSE': rmse_lgb},
    {'Target': 'Daily Case Count',           'Model': 'CatBoost', 'Task': 'Regression',
     'Accuracy': r2_cat, 'F1': np.nan, 'Precision': np.nan, 'Recall': np.nan,
     'MAE': mae_cat, 'RMSE': rmse_cat},
])

# Format for display
display_cols = ['Target', 'Model', 'Task', 'Accuracy', 'F1', 'Precision', 'Recall']
grand_display = grand_results[display_cols].copy()
for col in ['Accuracy', 'F1', 'Precision', 'Recall']:
    grand_display[col] = grand_display[col].apply(
        lambda x: f'{x:.4f}' if pd.notnull(x) else '—'
    )

print('=' * 90)
print('  GRAND COMPARISON: ALL 5 TARGETS × 2 MODELS (10 runs)')
print('  Note: For Regression, Accuracy column shows R² score')
print('=' * 90)
print(grand_display.to_string(index=False))

In [ ]:
# ── Regression metrics separately (since they use different scales) ──
print('\n--- Regression Target (Daily Case Count) ---')
reg_display = grand_results[grand_results['Task'] == 'Regression'][['Target', 'Model', 'MAE', 'RMSE', 'Accuracy']].copy()
reg_display = reg_display.rename(columns={'Accuracy': 'R²'})
for col in ['MAE', 'RMSE', 'R²']:
    reg_display[col] = reg_display[col].apply(lambda x: f'{x:.4f}' if pd.notnull(x) else '—')
print(reg_display.to_string(index=False))

In [ ]:
# ── Grand visual: Classification targets (Accuracy & F1) ──

clf_results = grand_results[grand_results['Task'] == 'Classification'].copy()
clf_results['label'] = clf_results['Model'] + '\n' + clf_results['Target'].str.split('(').str[0].str.strip()

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(clf_results))
w = 0.35

bars1 = ax.bar(x - w/2, clf_results['Accuracy'], width=w, label='Accuracy',
               color='steelblue', edgecolor='black', linewidth=0.3)
bars2 = ax.bar(x + w/2, clf_results['F1'], width=w, label='F1 (weighted)',
               color='coral', edgecolor='black', linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(clf_results['label'], fontsize=8, rotation=30, ha='right')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('All Classification Models — Accuracy & F1 Comparison', fontweight='bold', fontsize=13)
ax.legend(loc='upper right')

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Winner analysis ──
print('\n' + '=' * 70)
print('  WINNER ANALYSIS')
print('=' * 70)

# For each target, find the best model by F1 (classification) or R² (regression)
for target in grand_results['Target'].unique():
    subset = grand_results[grand_results['Target'] == target]
    
    if subset['Task'].iloc[0] == 'Classification':
        best = subset.loc[subset['F1'].idxmax()]
        print(f'\n  {target}')
        print(f'    Winner: {best["Model"]}  |  F1={best["F1"]:.4f}  |  Acc={best["Accuracy"]:.4f}')
        diff = abs(subset['F1'].values[0] - subset['F1'].values[1])
        print(f'    Margin: {diff:.4f} F1 points')
    else:
        best = subset.loc[subset['Accuracy'].idxmax()]  # R²
        print(f'\n  {target}')
        print(f'    Winner: {best["Model"]}  |  R²={best["Accuracy"]:.4f}  |  MAE={best["MAE"]:.2f}')
        diff = abs(subset['Accuracy'].values[0] - subset['Accuracy'].values[1])
        print(f'    Margin: {diff:.4f} R² points')

print(f'\n{"=" * 70}')

---
## 13. Key Takeaways

### Models
- **LightGBM** and **CatBoost** are both state-of-the-art gradient boosting frameworks
- LightGBM is typically faster to train; CatBoost often edges ahead on categorical-heavy data
- Both significantly outperform basic Random Forest / vanilla XGBoost

### Targets Summary
| Target | What we predicted | Practical use |
|--------|-------------------|---------------|
| `LAW_CAT_CD` | Crime severity level | Priority dispatch |
| `CRM_ATPT_CPTD_CD` | Whether crime was completed | Prevention strategy |
| `BORO_NM` | Which borough | Geographic allocation |
| `OFNS_DESC` | Type of offense | Specialised unit routing |
| `Daily Count` | Number of crimes per day | Staffing & scheduling |